# RLI 17.00 — Pit Lane Repairs
## Assignment Report

---

## Part 01 — Converting Q-Table to Vanilla DQN (60%)

### Conceptual Approach

The original `Pyrace_RL_QTable.py` uses a tabular Q-Learning approach where a Q-table stores expected cumulative rewards for every (state, action) pair. While effective for small discrete state spaces, this approach does not scale to larger or continuous environments.

In `Pyrace_RL_DQN.py`, we replace the Q-table with a **Deep Q-Network (DQN)** — a neural network that approximates the Q-function. The key changes are:

1. **Neural Network (DQN class):** A feedforward network with two hidden layers (128 neurons each, ReLU activation) that takes a 5-dimensional state (radar distances) as input and outputs Q-values for each of the 3 possible actions (accelerate, turn left, turn right).

2. **Experience Replay (ReplayMemory class):** Instead of learning from each transition immediately, we store transitions `(state, action, reward, next_state, done)` in a replay buffer and sample random mini-batches for training. This breaks temporal correlations and stabilizes learning.

3. **Epsilon-Greedy Exploration:** The agent starts with high exploration (epsilon ≈ 1.0) and gradually shifts toward exploitation as it learns. Epsilon decays over episodes.

### Architecture Diagram

```
Input (5 radar distances)
        │
   [Linear 5 → 128]
        │
      [ReLU]
        │
   [Linear 128 → 128]
        │
      [ReLU]
        │
   [Linear 128 → 3]  (Q-values for 3 actions)
        │
   Output: Q(s, a) for each action
```

### What Changed from QTable

| Component | Q-Table | DQN |
|-----------|---------|-----|
| Value storage | NumPy array (`q_table`) | Neural network (`policy_net`) |
| Update rule | Direct Q-value update: `Q[s,a] += α(r + γ·max(Q[s']) - Q[s,a])` | Gradient descent on MSE loss between predicted and target Q-values |
| Memory | None (online learning) | Experience replay buffer (10,000 transitions) |
| State handling | `state_to_bucket()` discretization | Raw state fed directly to the network |
| Generalization | None — each state independent | NN generalizes across similar states |

### Key Design Decisions

- **No target network** was used, as specified in the assignment. This is a "vanilla" DQN with basic experience replay only.
- **No changes were made to `gym_race/`** for Part 01 — the original environment (`Pyrace-v1`) is used as-is.
- The `simulate()` function was the primary focus of the refactoring, as suggested.

---
## Part 02 — Improvements to the Model (40%)

### Overview of Improvements

Three key improvements were made to the environment, implemented in `pyrace_2d_v3.py` and `race_env_v3.py`, registered as **Pyrace-v3**:

### 1. Continuous Observations (instead of discrete)

**Problem:** In the original `pyrace_2d.py`, the `observe()` method discretizes radar distances:
```python
ret[i] = int(r[1] / 20)  # rounds to 20-pixel intervals → values 0–10
```
This throws away fine-grained spatial information. The agent cannot distinguish between distances of 1 and 19 pixels — both map to bucket 0.

**Solution:** In `pyrace_2d_v3.py`, we pass raw continuous radar distances:
```python
ret[i] = float(r[1])  # pristine continuous distances, 0.0 – 200.0
```
The observation space in `race_env_v3.py` is updated accordingly:
```python
self.observation_space = spaces.Box(
    np.array([0.0, 0.0, 0.0, 0.0, 0.0]),
    np.array([200.0, 200.0, 200.0, 200.0, 200.0]),
    dtype=np.float32
)
```
This gives the agent much richer spatial information to learn from.

### 2. New Action: BRAKE (action == 3)

**Problem:** The original action space only has 3 actions:
- `0`: Accelerate (`speed += 2`)
- `1`: Turn left (`angle += 5`)
- `2`: Turn right (`angle -= 5`)

There is no way for the agent to actively slow down. Deceleration only happens passively through friction (`speed -= 0.5` every tick). This limits the agent's ability to navigate tight corners.

**Solution:** A 4th action was added in `pyrace_2d_v3.py`:
```python
elif action == 3: self.car.speed -= 3  # explicit braking
```
The speed floor was also lowered from `1` to `0` in `Car.update()` so that braking can actually bring the car to a full stop, giving the agent the ability to stop before a wall and reassess.

The action space is now `Discrete(4)`.

### 3. Dense Reward Function

**Problem:** The original reward function is extremely sparse:
- Crash: `reward = -10000 + distance`
- Goal (lap complete): `reward = 10000`
- Everything else: `reward = 0`

The agent gets zero feedback during normal driving. It only learns when it crashes or (rarely) completes a lap. This makes learning very slow since the signal is too infrequent.

**Solution:** In `pyrace_2d_v3.py`, we add a dense intermediate reward:
```python
else:
    reward = self.car.speed * 0.1  # reward for staying alive and moving fast
```
This encourages the agent to:
- **Stay alive** (dead cars get no positive reward)
- **Maintain speed** (higher speed → higher reward per step)

The crash and goal rewards are preserved, but now the agent gets continuous feedback every step.

### Possible Further Improvements (Discussion)

- **Checkpoint-based reward:** The commented-out checkpoint reward (`2000 - time_spent`) could be combined with the speed reward to incentivize progress around the track, not just raw speed.
- **Distance-to-next-checkpoint reward:** Using `cur_distance` to the next checkpoint as a shaping reward would directly guide the agent toward the racing line.
- **Penalizing excessive turning:** Adding a small negative reward for turning actions could encourage smoother driving.
- **Normalized observations:** Dividing radar distances by 200 (the max) to get values in [0, 1] could help neural network training convergence.

---
## Bonus — Migration to Stable-Baselines3 (30%)

### Approach

Since the `gym_race` environment is fully compatible with OpenAI Gymnasium, migrating to **Stable-Baselines3 (SB3)** is straightforward. We use **PPO (Proximal Policy Optimization)**, a state-of-the-art Policy Gradient algorithm.

Two implementations are provided:
- `Pyrace_RL_SB3.py` — PPO on the original `Pyrace-v1` (discrete observations, 3 actions)
- `Pyrace_RL_SB3_v3.py` — PPO on the improved `Pyrace-v3` (continuous observations, 4 actions)

### Why PPO?

- **Handles both discrete and continuous** action/observation spaces
- **Stable training** — uses clipped surrogate objective to prevent destructive large policy updates
- **Sample efficient** compared to vanilla Policy Gradients
- **Simple API** via SB3: model creation, training, saving, and evaluation in ~10 lines of code

### Code Structure

```python
from stable_baselines3 import PPO

env = gym.make('Pyrace-v3')
model = PPO('MlpPolicy', env, verbose=1)
model.learn(total_timesteps=50_000)
model.save('ppo_pyrace')

mean_reward, std_reward = evaluate_policy(model, env, n_eval_episodes=5)
```

### Results

With 50,000 timesteps the agent achieves modest performance, as PPO requires more training time for this environment. The primary goal of this bonus section was to demonstrate that the Gymnasium-compatible environment integrates seamlessly with advanced RL frameworks like SB3, opening the door to experimenting with algorithms like DDPG, SAC, or A2C with minimal code changes.

### Possible Extensions

- **DDPG / SAC** for continuous action spaces (e.g., continuous steering angle and throttle)
- **Longer training** (500k+ timesteps) for better convergence
- **Hyperparameter tuning** (learning rate, batch size, n_steps)
- **VecEnv wrappers** for parallel training across multiple environment instances

---
## File Overview

| File | Description |
|------|-------------|
| `Pyrace_RL_QTable.py` | Professor's original Q-Table starter code |
| `Pyrace_RL_DQN.py` | **Part 01** — Vanilla DQN implementation |
| `Pyrace_RL_SB3.py` | **Bonus** — SB3 PPO on Pyrace-v1 |
| `Pyrace_RL_SB3_v3.py` | **Bonus** — SB3 PPO on Pyrace-v3 |
| `Pyrace_performance_analysis.ipynb` | Performance analysis notebook |
| `gym_race/envs/pyrace_2d.py` | Original environment (unchanged) |
| `gym_race/envs/race_env.py` | Original Gym wrapper (unchanged) |
| `gym_race/envs/pyrace_2d_v3.py` | **Part 02** — Improved environment |
| `gym_race/envs/race_env_v3.py` | **Part 02** — Improved Gym wrapper |
| `gym_race/__init__.py` | Registers Pyrace-v1 and Pyrace-v3 |
| `gym_race/envs/__init__.py` | Module exports |